# Build Aspect Based Sentiment analysis cho bộ dữ liệu ABSA Tiki book review

In [ ]:
# Truy cập drive để lấy dữ liệu
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Tải các thư viện cần thiết
## !pip install transformers
## !pip install torchvision
## !pip install Dataset

## Import mấy cái này nếu cần

!pip install underthesea

In [ ]:
# Import các thư viện cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils import resample
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from datasets import Dataset
from underthesea import word_tokenize
print("Libraries imported successfully.")

Libraries imported successfully.


## chạy lần lượt gán nhãn của sentiment đến content, physical, price ...
- Mỗi vòng lặp như vậy sẽ đi gán lại vào bộ dữ liệu tiki

In [ ]:
# Load bộ dữ liệu training
df = pd.read_csv('/content/drive/MyDrive/Data Training/train_data_clean.csv')

# Fill các dữ liệu thiếu
if 'content' in df.columns:
    df['content'] = df['content'].fillna("")

display(df.head())

,review_id,rating,review_title,product_name,category,content,content_raw,sentiment_llm,as_content,as_physical,as_price,as_packaging,as_delivery,as_service
0,14357443,4,Hài lòng,Sách Xứ Cát,Sách văn học,"Sản phẩm thật sự chất lượng, và Tiki giao hàng...","Sản phẩm thật sự chất lượng, và Tiki giao hàng...",1,NaN,NaN,1.0,NaN,2.0,NaN
1,16749123,1,Rất không hài lòng,Bộ sách Làm Giàu Từ Chứng Khoán (How To Make M...,Sách kinh tế,không phù hợp đầu tư dài hạn,Ko phù hợp đầu tư dài hạn,0,NaN,NaN,NaN,NaN,NaN,NaN
2,897166,1,Không nên mua,Sách Predictably Irrational : The Hidden Force...,Reference,"Cuốn sách không đúng khổ ghi trên thông tin, k...","Cuốn sách không đúng khổ ghi trên thông tin, k...",0,NaN,2.0,NaN,NaN,NaN,NaN
3,9105241,3,Bình thường,Siêu Cò – How To Be A Power Connector,Sách kinh tế,"Sách hay, nội dung hấp dẫn, thiết thực. Chấm 5...","Sách hay, nội dung hấp dẫn, thiết thực.\nChấm ...",1,2.0,NaN,NaN,NaN,2.0,NaN
4,6820537,4,Hài lòng,Sách Elon Musk: How The Billionaire CEO Of Spa...,Biographies & Memoirs,sách chất lượng in chưa được cao,sách chất lượng in chưa được cao,1,NaN,1.0,NaN,NaN,NaN,NaN


In [ ]:
MODEL_NAME = "vinai/phobert-base"
TEXT_COL = "content"
LABEL_COL = "sentiment_llm"   # đổi lần lượt thành các aspect khác

In [ ]:
# Chia tập balanced thành train (80%) và temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[LABEL_COL],
    random_state=42
)

# Chia tiếp temp thành val (10%) và test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df[LABEL_COL],
    random_state=42
)

print(f"Kích thước tập train: {len(train_df)}")
print(f"Kích thước tập validation: {len(val_df)}")
print(f"Kích thước tập test: {len(test_df)}")

print("\nPhân bố nhãn trong tập Train:")
display(train_df[LABEL_COL].value_counts())

print("\nPhân bố nhãn trong tập Validation:")
display(val_df[LABEL_COL].value_counts())

print("\nPhân bố nhãn trong tập Test:")
display(test_df[LABEL_COL].value_counts())

Kích thước tập train: 8556
Kích thước tập validation: 1070
Kích thước tập test: 1070

Phân bố nhãn trong tập Train:


,count
sentiment_llm,
0,4486
2,2675
1,1395



Phân bố nhãn trong tập Validation:


,count
sentiment_llm,
0,561
2,335
1,174



Phân bố nhãn trong tập Test:


,count
sentiment_llm,
0,561
2,334
1,175


In [ ]:
# Initialize model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3    # 3 classes if labels are 0,1,2
)

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def vietnamese_word_segmenter(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception as e:
        return str(text)

# 2. Split the dataset again
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df[LABEL_COL], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df[LABEL_COL], random_state=42)

# 3. Apply word segmentation on the new combined column
train_df['segmented_content'] = train_df['content'].apply(vietnamese_word_segmenter)
val_df['segmented_content'] = val_df['content'].apply(vietnamese_word_segmenter)
test_df['segmented_content'] = test_df['content'].apply(vietnamese_word_segmenter)

# 4. Create Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df[['content', LABEL_COL]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['content', LABEL_COL]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['content', LABEL_COL]], preserve_index=False)

display(train_df[['content', 'segmented_content']].head())


,content,segmented_content
7414,"Nhỏ hơn bàn tay tí, chưa đọc ,khi nào xong fix...","Nhỏ hơn bàn_tay tí , chưa đọc , khi nào xong f..."
7937,Không bọc bao bì cho cuốn sách,Không bọc bao_bì cho cuốn sách
5186,Sách hay dành cho nhà lãnh đạo.,Sách hay dành cho nhà_lãnh_đạo .
1509,"Mình thấy sách hay, ý nghĩa cho cả trẻ em và n...","Mình thấy sách hay , ý_nghĩa cho cả trẻ_em và ..."
760,Sách được đóng gói cẩn thận. Lại thêm một cuốn...,Sách được đóng_gói cẩn_thận . Lại thêm một cuố...


In [ ]:
train_df.to_csv('/content/drive/MyDrive/Data Training/train_data_clean_segmented.csv', index=False)

In [ ]:
#Check dataframe
print("Train_Dataset:")
print(train_ds)

Train_Dataset:
Dataset({
    features: ['content', 'sentiment_llm'],
    num_rows: 8556
})


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['segmented_content'],
        padding='max_length',
        truncation=True,
        max_length=256
    )


# Keep only the necessary columns
train_subset = train_df[['segmented_content', 'sentiment_llm']]
val_subset = val_df[['segmented_content', 'sentiment_llm']]
test_subset = test_df[['segmented_content', 'sentiment_llm']]

# Convert to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_subset, preserve_index=False)
val_dataset = Dataset.from_pandas(val_subset, preserve_index=False)
test_dataset = Dataset.from_pandas(test_subset, preserve_index=False)

# Verify the train dataset
print("Train Dataset:")
print(train_dataset)




Train Dataset:
Dataset({
    features: ['segmented_content', 'sentiment_llm'],
    num_rows: 8556
})


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['segmented_content'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# Apply tokenization to all datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Remove the original text column
tokenized_train = tokenized_train.remove_columns(['segmented_content'])
tokenized_val = tokenized_val.remove_columns(['segmented_content'])
tokenized_test = tokenized_test.remove_columns(['segmented_content'])

# Verify
print("Tokenized Train Dataset Features:", tokenized_train.features)
print("\nFirst row of tokenized train dataset:")
print(tokenized_train[0])

Map:   0%|          | 0/8556 [00:00<?, ? examples/s]

Map:   0%|          | 0/1070 [00:00<?, ? examples/s]

Map:   0%|          | 0/1070 [00:00<?, ? examples/s]

Tokenized Train Dataset Features: {'sentiment_llm': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}

First row of tokenized train dataset:
{'sentiment_llm': 1, 'input_ids': [0, 18552, 48, 3178, 10294, 4, 102, 987, 4, 26, 142, 954, 8520, 2094, 44, 423, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [ ]:
# các parameter để train
training_args = TrainingArguments(
    output_dir="./sentiment_class",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    logging_dir="./train_logs",
    logging_steps=100,
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# F1_score, Accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

# Rename the label column to 'labels' as expected by the Trainer
tokenized_train = tokenized_train.rename_column(LABEL_COL, "labels")
tokenized_val = tokenized_val.rename_column(LABEL_COL, "labels")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Restarting training with optimized parameters...")
trainer.train()

Restarting training with optimized parameters...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.801217,0.351490,0.854206,0.817920
2,0.585062,0.313335,0.874766,0.827149
3,0.431174,0.327362,0.881308,0.829341
4,0.288272,0.373601,0.878505,0.834603
5,0.222320,0.423390,0.879439,0.835011
6,0.174871,0.463164,0.877570,0.834787


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1608, training_loss=0.44946220646894985, metrics={'train_runtime': 368.5182, 'train_samples_per_second': 232.173, 'train_steps_per_second': 7.272, 'total_flos': 6753595206242304.0, 'train_loss': 0.44946220646894985, 'epoch': 6.0})

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Ensure the label column is named 'labels' for the trainer to recognize it as ground truth
tokenized_test_final = tokenized_test.rename_column(LABEL_COL, "labels")

# Predict using the renamed dataset
test_results = trainer.predict(tokenized_test_final)
pred_labels = np.argmax(test_results.predictions, axis=1)
true_labels = test_results.label_ids

# Check if true_labels exist before calculating metrics (using None in Python)
if true_labels is not None:
    test_acc = accuracy_score(true_labels, pred_labels)
    print(f"Độ chính xác (accuracy) trên tập test: {test_acc:.4f}")

    class_names = ["Tiêu cực", "Trung lập", "Tích cực"]
    print("Báo cáo phân loại trên tập test:")
    print(classification_report(true_labels, pred_labels, target_names=class_names))
else:
    print("Lỗi: Không tìm thấy nhãn thực tế (true labels) trong dataset.")

Độ chính xác (accuracy) trên tập test: 0.8748
Báo cáo phân loại trên tập test:
              precision    recall  f1-score   support

    Tiêu cực       0.94      0.92      0.93       561
   Trung lập       0.67      0.61      0.63       175
    Tích cực       0.86      0.94      0.90       334

    accuracy                           0.87      1070
   macro avg       0.82      0.82      0.82      1070
weighted avg       0.87      0.87      0.87      1070



In [ ]:
# Lưu lại mô hình đã được fine-tune (tốt nhất) và tokenizer
trainer.save_model("./best_phobert_model")
tokenizer.save_pretrained("./best_phobert_model")

# In ra kết quả mô hình tốt nhất trên tập validation
best_acc = trainer.state.best_metric
print(f"Độ chính xác tốt nhất trên tập validation: {best_acc:.4f}")
print("Checkpoint mô hình tốt nhất được lưu tại:", trainer.state.best_model_checkpoint)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Độ chính xác tốt nhất trên tập validation: 0.8794
Checkpoint mô hình tốt nhất được lưu tại: ./sentiment_class/checkpoint-1876


#Gán nhãn bộ dữ liệu tiki gốc

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from underthesea import word_tokenize
from tqdm.auto import tqdm

# 1. Tải mô hình và tokenizer
model_dir = "./best_phobert_model"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 2. Đọc dữ liệu
df_unlabel = pd.read_csv('/content/drive/MyDrive/tiki_reviews.csv')
# Xử lý giá trị rỗng
df_unlabel['content'] = df_unlabel['content'].fillna('')

# 3. Hàm tiền xử lý (tách từ)
def vietnamese_word_segmenter(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception:
        return str(text)

print("Đang tách từ...")
df_unlabel['segmented_content'] = df_unlabel['content'].apply(vietnamese_word_segmenter)

# 4. Tạo Pytorch Dataset để xử lý batch
class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0)
        }

dataset = ReviewDataset(df_unlabel['segmented_content'].tolist(), tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# 5. Dự đoán
all_preds = []
all_scores = []

print("Đang dự đoán cảm xúc...")
with torch.no_grad():
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)

        # Lấy nhãn có xác suất cao nhất
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        # Tính sentiment score (ví dụ: xác suất của lớp được chọn)
        scores = torch.max(probs, dim=1).values.cpu().numpy()

        all_preds.extend(preds)
        all_scores.extend(scores)

# 6. Gán nhãn và lưu kết quả
# Mapping (NEG: 0, NEU: 1, POS: 2) -> (Tiêu cực, Trung tính, Tích cực)
label_map = {0: 'Tiêu cực', 1: 'Trung tính', 2: 'Tích cực'}
df_unlabel['predicted_label_id'] = all_preds
df_unlabel['sentiment_label'] = df_unlabel['predicted_label_id'].map(label_map)
df_unlabel['sentiment_score'] = all_scores

# Lưu lại file
save_path = '/content/drive/MyDrive/tiki_reviews_with_sentiment.csv'
df_unlabel.to_csv(save_path, index=False)
print(f"Đã lưu kết quả tại: {save_path}")

display(df_unlabel[['content', 'sentiment_label', 'sentiment_score']].head(10))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Đang tách từ...
Đang dự đoán cảm xúc...


  0%|          | 0/2406 [00:00<?, ?it/s]

Đã lưu kết quả tại: /content/drive/MyDrive/tiki_reviews_with_sentiment.csv


,content,sentiment_label,sentiment_score
0,mấy cái drama trong cuốn sách này ngày nào côn...,Tích cực,0.995592
1,Cực kỳ hài lòng\r\nSản phẩm có chất lượng đúng...,Tích cực,0.997491
2,Tất cả mọi vấn đề mình đang gặp phải đều có tr...,Tiêu cực,0.915589
3,Cái hay ho thú vị của quyển sách này không nằm...,Tích cực,0.997657
4,Nội dung quá tuyệt vời cho tất cả intern nvien...,Tích cực,0.997556
5,Một cuốn sách đáng đọc không những cho *** bạn...,Tích cực,0.997538
6,Cảm giác như đang cầm trên tay 1 sự chắt lọc r...,Tích cực,0.997605
7,Lần đầu tiên mình đọc hết một cuốn sách trong ...,Tích cực,0.997575
8,"[Một cuốn sách thực sự rất ha, ý nghĩa và mình...",Tích cực,0.997641
9,"Sách rất hay, văn phong của tác giả phù hợp vớ...",Tích cực,0.997637
